# Quality Control for the processed parquet

In [1]:
import pandas as pd
import yaml
from pathlib import Path

# --- Setup ---
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

panel = pd.read_parquet("../processed_data/analysis_panel.parquet")
panel_ind = pd.read_parquet("../processed_data/indicators_panel.parquet")  # without Total, with indicators

In [2]:
# --- Basic structure ---
print("=== ANALYSIS PANEL ===")
print(f"Shape: {panel.shape[0]:,} rows x {panel.shape[1]} columns")
print(f"IS8 sectors: {sorted(panel['IS8_SECTOR'].unique())}")
print(f"Unique LADs: {panel['GEOGRAPHY_CODE'].nunique()}")
print(f"Years: {sorted(panel['YEAR'].unique())}")

print("\n=== INDICATORS PANEL ===")
print(f"Shape: {panel_ind.shape[0]:,} rows x {panel_ind.shape[1]} columns")
print(f"IS8 sectors: {sorted(panel_ind['IS8_SECTOR'].unique())}")
print(f"Unique LADs: {panel_ind['GEOGRAPHY_CODE'].nunique()}")
print(f"Years: {sorted(panel_ind['YEAR'].unique())}")

=== ANALYSIS PANEL ===
Shape: 30,800 rows x 53 columns
IS8 sectors: ['Advanced Manufacturing', 'Creative Industries', 'Defence', 'Digital and Technologies', 'Financial Services', 'Life Sciences', 'Professional and Business Services', 'Total']
Unique LADs: 350
Years: [np.int16(2015), np.int16(2016), np.int16(2017), np.int16(2018), np.int16(2019), np.int16(2020), np.int16(2021), np.int16(2022), np.int16(2023), np.int16(2024), np.int16(2025)]

=== INDICATORS PANEL ===
Shape: 26,950 rows x 63 columns
IS8 sectors: ['Advanced Manufacturing', 'Creative Industries', 'Defence', 'Digital and Technologies', 'Financial Services', 'Life Sciences', 'Professional and Business Services']
Unique LADs: 350
Years: [np.int16(2015), np.int16(2016), np.int16(2017), np.int16(2018), np.int16(2019), np.int16(2020), np.int16(2021), np.int16(2022), np.int16(2023), np.int16(2024), np.int16(2025)]


In [3]:
# --- Missing values ---
print("=== MISSING VALUES (indicators panel) ===")
print(panel_ind.isnull().sum().to_string())

=== MISSING VALUES (indicators panel) ===
YEAR                                0
GEOGRAPHY_CODE                      0
GEOGRAPHY_NAME                      0
IS8_SECTOR                          0
EMPLOYEES                        2450
BUSINESSES                       2450
gva_per_hour                      308
weekly_pay                        154
employment_rate                   462
unemployment_rate                1617
gdhi_per_head                     308
new_enterprises                   308
deaths_of_enterprises             308
active_enterprises                308
high_growth_enterprises           308
uk_exports                      26950
inward_fdi                      26950
outward_fdi                     26950
goverment_randd                 26950
public_transport_to_employer     5467
drive_to_employer                5467
cycle_to_employer                5467
broadband_availability            308
4g_area_coverage                  308
ks2_attainment                  16786
gcse_by_

In [4]:
print(panel_ind['lq_emp'].describe())
print(f"\nSample lq_emp values:")
print(panel_ind[panel_ind['GEOGRAPHY_NAME'] == 'Sheffield']['lq_emp'].head(5))

count       24500.0
mean       0.931461
std        3.160158
min             0.0
25%        0.259797
50%        0.574108
75%        0.989644
max      120.030762
Name: lq_emp, dtype: Float64

Sample lq_emp values:
1715     0.47333
1716    0.661367
1717         0.0
1718    0.777725
1719    1.180456
Name: lq_emp, dtype: Float64


In [5]:
import sys
sys.path.insert(0, "..")

from src.indicators import IndicatorBuilder

builder = IndicatorBuilder(config)

# Test LQ on raw panel (with Total rows)
test = builder.compute_location_quotient(panel.copy())
print(f"Rows after LQ: {test.shape[0]:,}")
print(f"lq_emp null count: {test['lq_emp'].isnull().sum():,}")
print(f"lq_emp sample (Sheffield, Adv Mfg, 2023):")
print(test[
    (test['GEOGRAPHY_NAME'] == 'Sheffield') &
    (test['IS8_SECTOR'] == 'Advanced Manufacturing') &
    (test['YEAR'] == 2023)
][['lq_emp']].values)

Rows after LQ: 26,950
lq_emp null count: 2,450
lq_emp sample (Sheffield, Adv Mfg, 2023):
[[0.49832895]]


In [6]:
test = panel.copy()
test = builder.compute_employment_share(test)
test = builder.compute_location_quotient(test)

print("Non-null lq_emp count:", test['lq_emp'].notna().sum())
print("Null lq_emp count:", test['lq_emp'].isna().sum())
print("\nlq_emp for Sheffield, Adv Mfg, 2023:")
print(test[
    (test['GEOGRAPHY_NAME'] == 'Sheffield') &
    (test['IS8_SECTOR'] == 'Advanced Manufacturing') &
    (test['YEAR'] == 2023)
][['IS8_SECTOR', 'lq_emp']])

Non-null lq_emp count: 24500
Null lq_emp count: 2450

lq_emp for Sheffield, Adv Mfg, 2023:
                   IS8_SECTOR    lq_emp
21315  Advanced Manufacturing  0.498329


In [7]:
test2 = panel.copy()

# replicate what compute_location_quotient does step by step
total = (
    test2[test2["IS8_SECTOR"] == "Total"]
    [["YEAR", "GEOGRAPHY_CODE", "EMPLOYEES"]]
    .rename(columns={"EMPLOYEES": "TOTAL_EMP"})
)

national = (
    test2[test2["IS8_SECTOR"] != "Total"]
    .groupby(["YEAR", "IS8_SECTOR"], as_index=False)["EMPLOYEES"]
    .sum()
    .rename(columns={"EMPLOYEES": "NAT_IS8_EMP"})
)

nat_total = (
    test2[test2["IS8_SECTOR"] == "Total"]
    .groupby("YEAR", as_index=False)["EMPLOYEES"]
    .sum()
    .rename(columns={"EMPLOYEES": "NAT_TOTAL_EMP"})
)

print(f"Total rows: {total.shape}")
print(f"National IS8 rows: {national.shape}")
print(f"National total rows: {nat_total.shape}")
print(f"\nSample total:\n{total.head(3)}")
print(f"\nSample national:\n{national.head(3)}")
print(f"\nSample nat_total:\n{nat_total.head(3)}")

Total rows: (3850, 3)
National IS8 rows: (77, 3)
National total rows: (11, 2)

Sample total:
    YEAR GEOGRAPHY_CODE  TOTAL_EMP
7   2015      E06000001      29850
15  2015      E06000002      59920
23  2015      E06000003      41600

Sample national:
   YEAR              IS8_SECTOR  NAT_IS8_EMP
0  2015  Advanced Manufacturing       853450
1  2015     Creative Industries      1359635
2  2015                 Defence        17685

Sample nat_total:
   YEAR  NAT_TOTAL_EMP
0  2015       28716755
1  2016       29205410
2  2017       29517910


In [8]:
# continue from previous cell
test3 = test2[test2["IS8_SECTOR"] != "Total"].copy()
print(f"Non-total rows before merge: {test3.shape}")

test3 = test3.merge(total, on=["YEAR", "GEOGRAPHY_CODE"], how="left")
print(f"After merging total: {test3['TOTAL_EMP'].notna().sum()} non-null TOTAL_EMP")

test3 = test3.merge(national, on=["YEAR", "IS8_SECTOR"], how="left")
print(f"After merging national: {test3['NAT_IS8_EMP'].notna().sum()} non-null NAT_IS8_EMP")

test3 = test3.merge(nat_total, on="YEAR", how="left")
print(f"After merging nat_total: {test3['NAT_TOTAL_EMP'].notna().sum()} non-null NAT_TOTAL_EMP")

test3["lq_emp"] = (
    (test3["EMPLOYEES"] / test3["TOTAL_EMP"]) /
    (test3["NAT_IS8_EMP"] / test3["NAT_TOTAL_EMP"])
)
print(f"\nNon-null lq_emp: {test3['lq_emp'].notna().sum()}")
print(f"\nSample:")
print(test3[
    (test3['GEOGRAPHY_NAME'] == 'Sheffield') &
    (test3['IS8_SECTOR'] == 'Advanced Manufacturing') &
    (test3['YEAR'] == 2023)
][['lq_emp', 'EMPLOYEES', 'TOTAL_EMP', 'NAT_IS8_EMP', 'NAT_TOTAL_EMP']])

Non-total rows before merge: (26950, 53)
After merging total: 24500 non-null TOTAL_EMP
After merging national: 26950 non-null NAT_IS8_EMP
After merging nat_total: 26950 non-null NAT_TOTAL_EMP

Non-null lq_emp: 24500

Sample:
         lq_emp  EMPLOYEES  TOTAL_EMP  NAT_IS8_EMP  NAT_TOTAL_EMP
21315  0.498329       3575     269950       832135       31312460


In [9]:
test4 = panel.copy()
print(f"Before emp_share: {test4.shape}, Total rows: {(test4['IS8_SECTOR']=='Total').sum()}")
test4 = builder.compute_employment_share(test4)
print(f"After emp_share: {test4.shape}, Total rows: {(test4['IS8_SECTOR']=='Total').sum()}")

Before emp_share: (30800, 53), Total rows: 3850
After emp_share: (30800, 54), Total rows: 3850


In [10]:
# Verify growth_emp for Sheffield Advanced Manufacturing
sheffield_emp = panel_ind[
    (panel_ind['GEOGRAPHY_NAME'] == 'Sheffield') &
    (panel_ind['IS8_SECTOR'] == 'Advanced Manufacturing')
][['YEAR', 'EMPLOYEES', 'growth_emp', 'cagr_emp']].dropna(subset=['growth_emp'])

print(sheffield_emp[['YEAR', 'EMPLOYEES', 'growth_emp', 'cagr_emp']].drop_duplicates(subset=['growth_emp']))

# Manual check: (2024 value - 2015 value) / 2015 value
emp_2015 = panel[
    (panel['GEOGRAPHY_NAME'] == 'Sheffield') &
    (panel['IS8_SECTOR'] == 'Advanced Manufacturing') &
    (panel['YEAR'] == 2015)
]['EMPLOYEES'].values[0]

emp_2024 = panel[
    (panel['GEOGRAPHY_NAME'] == 'Sheffield') &
    (panel['IS8_SECTOR'] == 'Advanced Manufacturing') &
    (panel['YEAR'] == 2024)
]['EMPLOYEES'].values[0]

manual_growth = (emp_2024 - emp_2015) / emp_2015
print(f"\nManual growth_emp: {manual_growth:.6f}")

      YEAR  EMPLOYEES  growth_emp  cagr_emp
1715  2015       3540   -0.011299 -0.001262

Manual growth_emp: -0.011299
